# 生物医学文献摘要器

## 练习目标（理念）

用 **Europe PMC** 的公开 API 按 PMCID 拉取全文 XML，抽出标题与摘要（abstract），再交给本地 **Ollama** 模型（如 `llama3.2`）生成面向学生/研究者的要点摘要。

- **输入**：PMCID（例如 `PMC1234567`）
- **输出**：文章标题 + 模型生成的要点摘要（Markdown）
- **技术点**：HTTP 抓取、BeautifulSoup 解析 XML、正则清洗文本、Ollama chat API

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 外部数据 → 再交给 LLM | Europe PMC XML → `messages` |
| system / user prompt | `sys_prompt` 定角色，user 放标题+摘要 |
| 本地模型 | `ollama.chat(model=MODEL, ...)` |

## 怎么跑

1. 本机已安装并启动 Ollama，且已 `ollama pull llama3.2`
2. 从上到下运行单元格；在调用格填入真实 PMCID
3. 需要：`requests`、`bs4`、`tqdm`、`loguru`、`ollama` 等依赖


In [ ]:
# ========== 导入：网页抓取、XML 解析、本地 Ollama ==========

# 标准库 re：用正则表达式清洗摘要里的引用标记、多余空白等
import re
# 标准库 pprint：漂亮打印（pretty-print）复杂结构，便于调试
import pprint
# 从 pprint 再导入 pformat：把对象格式化成字符串（本笔记本里可能备用）
from pprint import pformat
# 第三方 requests：发 HTTP GET，从 Europe PMC 拉 XML
import requests
# 标准库 functools：提供 wraps，用来写装饰器时保留原函数元信息
import functools
# typing：List / Tuple / Dict / Any，给函数签名做类型标注，方便阅读
from typing import List, Tuple, Dict, Any

# tqdm：进度条（本文件导入后未必处处用到，保留原依赖）
from tqdm import tqdm
# loguru：更友好的日志；@logger.catch 可自动捕获异常并记录
from loguru import logger
# BeautifulSoup：解析 HTML/XML；这里别名为 bs，后面解析 Europe PMC 的 XML
from bs4 import BeautifulSoup as bs

# IPython 展示工具：在笔记本里显示 HTML / Markdown
from IPython.display import display, HTML, Markdown

# ollama：Python 客户端，调用本机 Ollama 的 chat 接口
import ollama


## 工具函数：请求容错与 XML 拉取

下面先定义装饰器捕获网络错误，再定义从 URL 取 XML 的函数。


In [ ]:
# ========== 装饰器：捕获 requests 网络异常，失败时返回 None ==========

def catch_request_error(func):
    """
    Wrapper func to catch request errors and return None if an error occurs.
    Used as a decorator.
    """
    # wraps：让 wrapper 看起来仍像原来的 func（名字、docstring 等）
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        try:
            # 正常就直接执行被装饰的函数
            return func(*args, **kwargs)
        except requests.RequestException as e:
            # 网络类错误（超时、连接失败等）打印后返回 None，避免整条流水线崩溃
            # 报错文案保持英文（影响行为的字符串不翻译）
            print(f"Request error in {func.__name__}: {e}")
            return None
    return wrapper


In [ ]:
# ========== 从 Europe PMC URL 拉取并解析 XML ==========

# 叠两个装饰器：先 catch_request_error（网络失败→None），再 logger.catch（记日志）
@catch_request_error
@logger.catch
def get_xml_from_url(url: str) -> bs:
    """
    Fetches the XML content from Europe PMC website.

    Args:
        url (str): Europe PMC's production url to fetch the XML from.

    Returns:
        soup (bs4.BeautifulSoup): Parsed XML content.
    """
    # GET 请求下载全文 XML
    response = requests.get(url)
    # 非 2xx 时抛 HTTPError，交给上面的装饰器处理
    response.raise_for_status() #check for request errors
    # 用 BeautifulSoup 按 XML 解析字节内容，返回 soup 对象
    return bs(response.content, "xml")  


In [ ]:
# ========== 文本清洗 + 从 XML soup 提取标题与摘要 ==========

def clean_text(text:str) -> str:
    """
    This function cleans a text by filtering reference patterns in text, 
    extra whitespaces, escaped latex-style formatting appearing in text body instead of predefined latex tags

    Args: 
    text(str): The text to be cleaned
    
    Returns: 
    tex(str): The cleaned text 
    
    """
   
    # 去掉类似 LaTeX 的花括号内容（soup 已滤过一部分，正文里仍可能残留）
    text = re.sub(r"\{.*?\}", "", text)  # Matches and removes anything inside curly braces {}
    # 去掉反斜杠开头的命令名，如 \\alpha 一类残留
    text = re.sub(r"\\[a-zA-Z]+", "", text)  # Matches and removes characters that appears with numbers
    
    # 去掉文献引用标记，如 [34] 或 [1,2,3]
    text = re.sub(r"\[\s*(\d+\s*(,\s*\d+\s*)*)\]", "", text)
    
    # 把连续空白压成单个空格，并去掉首尾空白
    text = re.sub(r"\s+", " ", text).strip()
    
    return text


def fetch_article_abstract(soup: bs) -> Tuple[str, str]:
    """
    Extracts the abstract text from the XML soup.

    Args:
        soup (bs4.BeautifulSoup): Parsed XML content.
    Returns:
        Tuple(article_title (str), abstract_text (str)): A tuple of the article's title and its extracted abstract text.
    """
    # soup 为 None（上游请求失败）时给占位标题，摘要为空
    if soup is None:
        return "No XML found", ""
    # 找 <article-title>；找不到就用英文占位串（保留原字符串）
    article_title = soup.find("article-title").get_text(strip=True) if soup.find("article-title") else "No Title Found for this article"

    # 找 <abstract> 节点
    abstract_tag = soup.find("abstract")
    if abstract_tag:
        # 摘要里每个 <p> 取文本 → clean_text → 用空格拼成一段
        abstract_text = ' '.join([clean_text(p.get_text(strip=True)) for p in abstract_tag.find_all("p") if p.get_text(strip=True)])
    else:
        # 没有 abstract 标签就返回空字符串
        abstract_text = ""
    return article_title, abstract_text


In [ ]:
# ========== system prompt：定模型角色与摘要结构（发给模型的英文指令不翻译）==========

sys_prompt = """You are an expert in biomedical text mining and information extraction. 
You excel at breaking down complex articles into digestible contents for your audience. 
Your audience can comprise of students, early researchers and professionals in the field.
Summarize the key findings in the following article [ARTICLE] .
Your summary should provide crucial points covered in the paper that helps your diverse audience quickly understand the most vital information. 
Crucial points to consider:
- Main objectives of the study
- Key findings and results
- Methodologies used
- Implications of the findings(if any)
- Any limitations or future directions mentioned

Format: Provide your summary in bullet points highlighting key areas followed with a  concise paragraph that encapsulates the results of the paper.

The tone should be professional and clear.

"""


In [ ]:
# ========== 模型名常量：须与本机 ollama list 里的名字一致 ==========
MODEL = "llama3.2"


In [ ]:
# ========== 组装 Chat messages：system + user（标题与摘要塞进 user）==========

def build_message(article_title: str, abstract_text: str, sys_prompt:str=sys_prompt) -> List[Dict[str, str]]:
    """
    Constructs the message payload for the LLM.

    Args:
        article_title (str): The title of the article.
        abstract_text (str): The abstract text of the article.

    Returns:
        List[Dict[str, str]]: A list of message dictionaries for the LLM.
    """
    # user prompt 保留英文：这是发给模型的指令与内容，改译会改变回答行为
    user_prompt = f"""You are looking at an article with title:  {article_title}. 
    The article's abstract is as follows: \n{abstract_text}.
    Summarise the article. Start your summary by providing a short sentence on what the article is about 
    and then a bulleted list of the key points covered in the article.
"""
    # OpenAI 风格 messages 列表：先 system 定规矩，再 user 给具体文章
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_prompt}
    ]
    return messages


In [ ]:
# ========== 调用本地 Ollama：同步 chat，取出助手回复正文 ==========

def generate_response(messages, model=MODEL):
    # ollama.chat：把 messages 发给本地模型；返回结构里含 message.content
    response = ollama.chat(model=model, messages=messages)
    # 只返回助手文本，供后面 Markdown 展示
    return response["message"]["content"]


In [ ]:
# ========== 主流程：校验 PMCID → 拉 XML → 抽摘要 → 调 LLM → 展示 ==========

def display_reponse(article_id:str):
    # 用正则校验 PMCID：必须以 PMC 开头，后接 5–8 位数字
    if article_id and not re.match(r"^PMC\d{5,8}$", article_id):
        # 校验失败文案保持英文（原代码行为）
        raise ValueError("Please check the length/Format of the provided Article ID. It should start with 'PMC' followed by 5 to 8 digits, e.g., 'PMC1234567'.")
    # Europe PMC fullTextXML REST 端点；article_id 拼进路径
    url = f"https://www.ebi.ac.uk/europepmc/webservices/rest/{article_id}/fullTextXML"
    # 下载并解析 XML
    soup = get_xml_from_url(url)
    # 抽出标题与摘要文本
    article_title, abstract_text = fetch_article_abstract(soup)
    # 拼成 LLM messages
    messages = build_message(article_title, abstract_text)
    # 本地模型生成摘要
    response = generate_response(messages)

    # 在笔记本里用 Markdown 展示标题与模型回答（展示用英文标签保留）
    display(Markdown(f"### Article Title: {article_title}"))
    display(Markdown(f"### LLM Response: \n{response}"))


In [ ]:
# ========== 试跑：把 PMCID 换成你想摘要的文章 ==========
# 示例 PMCID；改这里的字符串即可测另一篇
display_reponse("PMC7394925")


In [ ]:
# 再试一个 PMCID（同上，只是文章不同）
display_reponse("PMC12375411")
